# Notebook 01: Alkene VQE Simulation
## Quantum Simulation of Alkenes via PySCF + OpenFermion + PennyLane

**Molecules:** Ethylene (C₂H₄)  
**Methods:** Hartree-Fock → CCSD → VQE (UCCSD ansatz, Adam optimizer)  
**Mapping:** Jordan-Wigner and Bravyi-Kitaev  
**Target:** Chemical accuracy (|ΔE| < 1.6 mHa vs FCI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/quantum-alkene-alkyne-pyscf/blob/main/notebooks/01_alkene_vqe_simulation.ipynb)

In [ ]:
# ── CELL 1: Install dependencies (run once in Colab) ─────────────────
import sys, subprocess
pkgs = [
    'pyscf>=2.5',
    'openfermion>=1.6',
    'openfermionpyscf>=0.5',
    'pennylane>=0.38',
    'tqdm',
    'seaborn',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Dependencies installed.')

In [ ]:
# ── CELL 2: Imports ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
from tqdm.auto import trange
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

from pyscf import gto, scf, ci, cc
from openfermion.chem import MolecularData
from openfermion import get_fermion_operator, jordan_wigner, bravyi_kitaev
from openfermion.utils import count_qubits
# freeze_orbitals moved in newer openfermion — handle both paths
try:
    from openfermion.transforms import freeze_orbitals
except ImportError:
    from openfermion.utils import freeze_orbitals
from openfermionpyscf import run_pyscf
import pennylane as qml
from pennylane import qchem
print(f'PennyLane {qml.__version__} loaded.')
print('All imports successful.')

In [ ]:
# ── CELL 3: Compatibility shim — PennyLane Hamiltonian builder ────────
# qml.Hamiltonian is deprecated in PennyLane ≥ 0.38; use qml.ops.LinearCombination
# qml.operation.Tensor is deprecated; use qml.prod()

import pennylane.version as _plv
_PL_NEW = tuple(int(x) for x in qml.__version__.split('.')[:2]) >= (0, 38)

def _pauli_gate(name, wire):
    return {'X': qml.PauliX, 'Y': qml.PauliY, 'Z': qml.PauliZ}[name](wire)

def openfermion_to_pennylane(qubit_op):
    """Convert OpenFermion QubitOperator to a PennyLane Hamiltonian.
    Compatible with both legacy (< 0.38) and modern (>= 0.38) PennyLane APIs.
    """
    coeffs, ops = [], []
    for term, coeff in qubit_op.terms.items():
        if abs(np.real(coeff)) < 1e-15:
            continue
        coeffs.append(np.real(coeff))
        if not term:
            ops.append(qml.Identity(0))
        else:
            gates = [_pauli_gate(p, i) for i, p in term]
            ops.append(gates[0] if len(gates) == 1 else qml.prod(*gates))
    if _PL_NEW:
        return qml.ops.LinearCombination(np.array(coeffs), ops)
    return qml.Hamiltonian(np.array(coeffs), ops)

print('Hamiltonian builder defined (API-compatible).')

In [ ]:
# ── CELL 4: Define Ethylene geometry and run classical references ─────
# Experimental geometry (Angstroms), STO-3G minimal basis
ethylene_geometry = [
    ('C', (0.000,  0.000,  0.000)),
    ('C', (0.000,  0.000,  1.339)),
    ('H', (0.000,  0.926, -0.546)),
    ('H', (0.000, -0.926, -0.546)),
    ('H', (0.000,  0.926,  1.885)),
    ('H', (0.000, -0.926,  1.885)),
]

BASIS = 'sto-3g'

eth = MolecularData(geometry=ethylene_geometry, basis=BASIS,
                    multiplicity=1, charge=0, description='ethylene')
eth = run_pyscf(eth, run_scf=True, run_ccsd=True, run_fci=True)

print('Ethylene classical reference energies (STO-3G):')
print(f'  HF   : {eth.hf_energy:.8f} Ha')
print(f'  CCSD : {eth.ccsd_energy:.8f} Ha')
print(f'  FCI  : {eth.fci_energy:.8f} Ha')
print(f'  Correlation energy (FCI-HF): {(eth.fci_energy - eth.hf_energy)*1000:.3f} mHa')
print(f'  n_electrons : {eth.n_electrons}')
print(f'  n_orbitals  : {eth.n_orbitals}')
print(f'  n_qubits (JW, full): {eth.n_qubits}')

In [ ]:
# ── CELL 5: Qubit Hamiltonian — JW vs BK + active space ───────────────
fermion_ham = get_fermion_operator(eth.get_molecular_hamiltonian())

# Full-space mappings
jw_ham_full = jordan_wigner(fermion_ham)
bk_ham_full = bravyi_kitaev(fermion_ham)
n_q_jw = count_qubits(jw_ham_full)
n_q_bk = count_qubits(bk_ham_full)

# Active space: freeze 3 lowest (core) orbitals → 8 active electrons, 4 active orbitals
# This gives a tractable 8-qubit circuit while preserving the π-system
active_fham = freeze_orbitals(fermion_ham, occupied=[0, 1, 2], virtual=[])
jw_active = jordan_wigner(active_fham)
bk_active = bravyi_kitaev(active_fham)
n_q_active_jw = count_qubits(jw_active)
n_q_active_bk = count_qubits(bk_active)
n_elec_active = eth.n_electrons - 6

print(f'Full JW: {n_q_jw} qubits | Full BK: {n_q_bk} qubits')
print(f'Active JW: {n_q_active_jw} qubits | Active BK: {n_q_active_bk} qubits')
print(f'Active electrons: {n_elec_active}')
print(f'Pauli terms (active JW): {len(list(jw_active.terms))}')

In [ ]:
# ── CELL 6: Build PennyLane Hamiltonian and circuit components ────────
H_pl = openfermion_to_pennylane(jw_active)
print(f'PennyLane Hamiltonian: {len(H_pl.coeffs)} terms')

singles, doubles = qchem.excitations(n_elec_active, n_q_active_jw)
hf_state = qchem.hf_state(n_elec_active, n_q_active_jw)
n_params = len(singles) + len(doubles)

print(f'HF state: {hf_state}')
print(f'Single excitations: {len(singles)}')
print(f'Double excitations: {len(doubles)}')
print(f'Total parameters  : {n_params}')

In [ ]:
# ── CELL 7: VQE circuit + Adam optimizer ─────────────────────────────
# Adam consistently outperforms plain GradientDescent for VQE:
# faster convergence, less sensitivity to step-size, avoids local minima.

dev = qml.device('default.qubit', wires=n_q_active_jw)

@qml.qnode(dev, diff_method='best')
def vqe_circuit(params):
    qml.BasisState(hf_state, wires=range(n_q_active_jw))
    qml.AllSinglesDoubles(
        params,
        wires=range(n_q_active_jw),
        hf_state=hf_state,
        singles=singles,
        doubles=doubles,
    )
    return qml.expval(H_pl)

# Initialise parameters with small random perturbation to break symmetry
np.random.seed(42)
params = np.random.uniform(-0.01, 0.01, n_params, requires_grad=True)

# Adam: lr=0.05 is a good default for molecular VQE
opt = qml.AdamOptimizer(stepsize=0.05, beta1=0.9, beta2=0.999)

MAX_ITER = 300
CONV_TOL = 1e-9
energies_jw = []

pbar = trange(MAX_ITER, desc='VQE (JW, ethylene)')
for step in pbar:
    params, energy = opt.step_and_cost(vqe_circuit, params)
    energies_jw.append(float(energy))
    pbar.set_postfix({'E': f'{energy:.8f}', 'ΔE_FCI': f'{abs(energy-eth.fci_energy)*1e3:.4f} mHa'})
    if step > 10 and abs(energies_jw[-1] - energies_jw[-2]) < CONV_TOL:
        print(f'Converged at step {step}')
        break

print(f'\nVQE  Energy : {energies_jw[-1]:.8f} Ha')
print(f'FCI  Energy : {eth.fci_energy:.8f} Ha')
print(f'CCSD Energy : {eth.ccsd_energy:.8f} Ha')
print(f'|VQE-FCI|   : {abs(energies_jw[-1]-eth.fci_energy)*1000:.4f} mHa')
print(f'Chemical accuracy (1.6 mHa): {"✓ ACHIEVED" if abs(energies_jw[-1]-eth.fci_energy)*1000 < 1.6 else "✗ not yet"}')

In [ ]:
# ── CELL 8: Publication-quality convergence plot ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: energy vs iteration
ax = axes[0]
ax.plot(energies_jw, color='steelblue', lw=2.0, label='VQE (Adam, JW)')
ax.axhline(eth.fci_energy, color='crimson', ls='--', lw=1.5, label=f'FCI = {eth.fci_energy:.6f} Ha')
ax.axhline(eth.ccsd_energy, color='seagreen', ls='-.', lw=1.5, label=f'CCSD = {eth.ccsd_energy:.6f} Ha')
ax.axhline(eth.hf_energy, color='slategray', ls=':', lw=1.5, label=f'HF = {eth.hf_energy:.6f} Ha')
ax.set_xlabel('Optimizer Iteration', fontsize=12)
ax.set_ylabel('Energy (Ha)', fontsize=12)
ax.set_title('VQE Convergence — Ethylene (STO-3G)', fontsize=13)
ax.legend(fontsize=9)

# Right: |error vs FCI| in mHa, log scale + chemical accuracy line
ax2 = axes[1]
errors_mHa = [abs(e - eth.fci_energy)*1000 for e in energies_jw]
ax2.semilogy(errors_mHa, color='steelblue', lw=2.0)
ax2.axhline(1.6, color='crimson', ls='--', lw=1.5, label='Chemical accuracy (1.6 mHa)')
ax2.set_xlabel('Optimizer Iteration', fontsize=12)
ax2.set_ylabel('|E_VQE − E_FCI| (mHa)', fontsize=12)
ax2.set_title('Error vs FCI (log scale)', fontsize=13)
ax2.legend(fontsize=9)

plt.suptitle('Ethylene · UCCSD-VQE · STO-3G · Active Space (8e/4o)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('ethylene_vqe_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# ── CELL 9: Qubit resource comparison across alkene homologs ──────────
# Analytical qubit counts for the series using STO-3G basis
# JW: 2 * n_orbitals; BK: ~same but slightly fewer for larger systems

alkene_series = [
    {'name': 'Ethylene (C₂H₄)',     'n_e': 16, 'n_o':  7,  'pi_bonds': 1},
    {'name': '1-Butene (C₄H₈)',     'n_e': 28, 'n_o': 13,  'pi_bonds': 1},
    {'name': '1,3-Butadiene (C₄H₆)','n_e': 28, 'n_o': 13,  'pi_bonds': 2},
    {'name': '1-Hexene (C₆H₁₂)',    'n_e': 40, 'n_o': 19,  'pi_bonds': 1},
    {'name': '1,3,5-Hexatriene',     'n_e': 40, 'n_o': 19,  'pi_bonds': 3},
]

rows = []
for m in alkene_series:
    jw_q  = m['n_o'] * 2
    bk_q  = int(np.ceil(np.log2(jw_q + 1))) * 2 + 2   # rough BK estimate
    # active-space heuristic: keep 2 * pi_bonds orbitals each side of HOMO
    as_o  = 4 * m['pi_bonds']
    as_q  = as_o * 2
    rows.append({'Molecule': m['name'],
                 'Electrons': m['n_e'], 'Orbitals': m['n_o'],
                 'JW Qubits': jw_q, 'BK Qubits': bk_q,
                 'Active-Space Q (NISQ)': as_q, 'π bonds': m['pi_bonds']})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(df))
w = 0.28
ax.bar(x - w,   df['JW Qubits'],          w, label='Full JW',  color='steelblue')
ax.bar(x,       df['BK Qubits'],          w, label='Full BK',  color='seagreen')
ax.bar(x + w,   df['Active-Space Q (NISQ)'], w, label='Active (NISQ)', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(df['Molecule'], rotation=18, ha='right', fontsize=9)
ax.set_ylabel('Qubit count', fontsize=11)
ax.set_title('Qubit Resource Scaling — Alkene Homologous Series', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('alkene_qubit_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 10: Summary table ────────────────────────────────────────────
summary = pd.DataFrame({
    'Method': ['HF', 'CCSD', 'FCI', 'VQE (UCCSD, Adam)'],
    'Energy (Ha)': [eth.hf_energy, eth.ccsd_energy, eth.fci_energy, energies_jw[-1]],
    '|ΔE_FCI| (mHa)': [
        abs(eth.hf_energy   - eth.fci_energy)*1000,
        abs(eth.ccsd_energy - eth.fci_energy)*1000,
        0.0,
        abs(energies_jw[-1] - eth.fci_energy)*1000,
    ],
})
summary['Chemical accuracy'] = summary['|ΔE_FCI| (mHa)'].apply(
    lambda x: '✓' if x < 1.6 else '✗')
print('Ethylene STO-3G — Method Comparison')
print(summary.to_string(index=False))
summary.to_csv('ethylene_benchmark.csv', index=False)
print('Saved to ethylene_benchmark.csv')

## Next Steps
- **Notebook 02**: Same workflow for alkynes (acetylene — stronger π-correlation)
- **Notebook 06**: ADAPT-VQE vs fixed UCCSD-VQE — adaptive ansatz growth saves CNOTs

## Key API Notes (PennyLane ≥ 0.38)
- `qml.Hamiltonian` → `qml.ops.LinearCombination`
- `qml.operation.Tensor` → `qml.prod()`
- `diff_method='best'` auto-selects adjoint differentiation on `default.qubit`
- Use `requires_grad=True` on numpy arrays for PennyLane autograd